# Notebook 6: Proposed Method V2 (chặn lặp khi giải mã)

**Lý do (từ phân tích lỗi V1):** một số câu dịch bị lặp vòng ("No, no, no...", "it's contagious, it's contagious..."): số câu có cụm 3 từ lặp từ 3 lần trở lên là 16 (A) và 31 (B) trên 4,199 bản ghi.

**Thay đổi duy nhất so với V1:** thêm `no_repeat_ngram_size=3` khi giải mã bước dịch. ASR, mô hình, beam (4) và mọi thứ khác giữ nguyên. Notebook dùng lại `fleurs_asr.csv` / `fleurs_all_outputs.csv` đã lưu, nên **không chạy lại ASR**.

Chấm điểm bằng cùng cách như notebook FLEURS: SacreBLEU, chrF++, khoảng tin cậy 95% bootstrap theo câu duy nhất, độ chênh dùng bootstrap cặp.

## 0. Môi trường

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install -U transformers accelerate sacrebleu rapidfuzz sentencepiece

## 1. Tham số

In [3]:
import os, re, json, time, gc
import numpy as np, pandas as pd, torch

ROOT      = '/content/drive/MyDrive/project_DPL'
OUT_DIR   = os.path.join(ROOT, 'results/fleurs_out')
ALL_CSV   = os.path.join(OUT_DIR, 'fleurs_all_outputs.csv')              # từ notebook FLEURS (ô 9)
MT_BASE   = os.path.join(ROOT, 'models/opus-mt-vi-en')
B_DIR     = os.path.join(ROOT, 'results/ft_out/B_100/best_model')
V2_DIR    = os.path.join(OUT_DIR, 'v2'); os.makedirs(V2_DIR, exist_ok=True)

SEED, MAX_SEC   = 42, 30
MT_BEAM, MT_BATCH, MT_MAXLEN = 4, 64, 128      # giống V1
NO_REPEAT = 3                                   # thay đổi của V2
N_BOOT = 1000
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

device: cuda | Tesla T4


## 2. Dữ liệu

In [4]:
def atomic_csv(d, path):
    tmp = path + '.tmp'; d.to_csv(tmp, index=False); os.replace(tmp, path)

df = pd.read_csv(ALL_CSV, keep_default_na=False)
for c in ['uid', 'split', 'fleurs_id', 'duration_sec', 'vi_script', 'en_script', 'asr_text', 'hyp_A', 'hyp_B']: assert c in df.columns, f'Thiếu cột {c}'
assert df.uid.is_unique
df['cluster'] = df.split + '|' + df.fleurs_id.astype(str)
print(f'{len(df):,} bản ghi | {df.cluster.nunique():,} câu duy nhất')

4,212 bản ghi | 1,978 câu duy nhất


## 3. Dịch lại với `no_repeat_ngram_size`
Có kiểm tra tái lập: dịch lại 300 câu đầu **không** bật chặn lặp và so với `hyp_A`/`hyp_B` của V1. Nếu khác nhiều thì cài đặt không giống V1 và kết quả V2 không so được.

In [6]:
assert DEVICE == 'cuda', 'Cần runtime GPU.'
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging
logging.set_verbosity_error()

tok = AutoTokenizer.from_pretrained(MT_BASE)

def translate(model, texts, **gen):
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i])); res = [''] * len(texts)
    for s in range(0, len(order), MT_BATCH):
        idx = order[s:s + MT_BATCH]
        enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=MT_MAXLEN).to('cuda')
        with torch.no_grad():
            out = model.generate(**enc, num_beams=MT_BEAM, max_new_tokens=160, **gen)
        for i, t in zip(idx, tok.batch_decode(out, skip_special_tokens=True)): res[i] = t.strip()
    return res

tm_path = os.path.join(V2_DIR, 'timing.json'); tm = json.load(open(tm_path)) if os.path.exists(tm_path) else {}
V2 = {}
for name, path, v1col in [('A', MT_BASE, 'hyp_A'), ('B', B_DIR, 'hyp_B')]:
    f = os.path.join(V2_DIR, f'{name}__asr__nr{NO_REPEAT}.csv'); ok = False
    if os.path.exists(f):
        try: ok = pd.read_csv(f, keep_default_na=False).uid.tolist() == df.uid.tolist()
        except Exception: ok = False
    if not ok:
        assert os.path.isfile(os.path.join(path, 'config.json')), f'Không thấy mô hình: {path}'
        model = AutoModelForSeq2SeqLM.from_pretrained(path).half().to('cuda').eval()
        translate(model, ['xin chào'] * 2)                                    # khởi động
        n = 300; chk = translate(model, df.asr_text.tolist()[:n])             # kiểm tra tái lập V1
        same = sum(a == b for a, b in zip(chk, df[v1col].tolist()[:n]))
        print(f'[{name}] tái lập V1: {same}/{n} câu giống hệt')
        torch.cuda.synchronize(); t0 = time.time()
        hyp = translate(model, df.asr_text.tolist(), no_repeat_ngram_size=NO_REPEAT); torch.cuda.synchronize(); dt = time.time() - t0
        atomic_csv(pd.DataFrame({'uid': df.uid, 'hyp': hyp}), f); tm[name] = {'s': dt, 'repro_same_of_300': same}
        json.dump(tm, open(tm_path, 'w')); del model; gc.collect(); torch.cuda.empty_cache()
        print(f'[{name}] V2 xong: {dt:.0f}s')
    V2[name] = pd.read_csv(f, keep_default_na=False).hyp.astype(str).values

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[A] tái lập V1: 296/300 câu giống hệt
[A] V2 xong: 55s


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[B] tái lập V1: 300/300 câu giống hệt
[B] V2 xong: 55s


## 4. Chấm điểm V1 so với V2
`ref` là hệ thống dùng để tính độ chênh: V2 so với V1 cùng mô hình; V1_B so với V1_A.

In [7]:
from sacrebleu.metrics import BLEU, CHRF
from collections import Counter
BLEU_M, CHRF_M = BLEU(), CHRF(word_order=2)
refs = df.en_script.astype(str).tolist()
SYS = {'V1_A': df.hyp_A.values, 'V2_A': V2['A'], 'V1_B': df.hyp_B.values, 'V2_B': V2['B']}
REFSYS = {'V1_A': None, 'V1_B': 'V1_A', 'V2_A': 'V1_A', 'V2_B': 'V1_B'}

def mt_stats(h):
    h = [str(x) for x in h]
    return (np.array(BLEU_M._extract_corpus_statistics(h, [refs]), dtype=float), np.array(CHRF_M._extract_corpus_statistics(h, [refs]), dtype=float))
STATS = {k: mt_stats(v) for k, v in SYS.items()}
sc_bleu = lambda s: BLEU_M._compute_score_from_stats(list(s)).score
sc_chrf = lambda s: CHRF_M._compute_score_from_stats(list(s)).score
def cluster_sum(M, cl): return pd.DataFrame(M).groupby(cl.values).sum().values
def ci(x): return np.percentile(x, [2.5, 97.5])

VIEWS = {'pooled_main': df.duration_sec <= MAX_SEC, 'pooled_all': pd.Series(True, index=df.index),
         'test_main': (df.split == 'test') & (df.duration_sec <= MAX_SEC)}

def eval_mt(view):
    m = VIEWS[view].values; cl = df.cluster[m]
    G = {k: (cluster_sum(STATS[k][0][m], cl), cluster_sum(STATS[k][1][m], cl)) for k in SYS}
    ncl = len(next(iter(G.values()))[0]); idx = np.random.default_rng(SEED).integers(0, ncl, size=(N_BOOT, ncl))
    point, boot = {}, {}
    for k, (b, c) in G.items():
        point[k] = (sc_bleu(b.sum(0)), sc_chrf(c.sum(0)))
        boot[k] = (np.array([sc_bleu(b[i].sum(0)) for i in idx]), np.array([sc_chrf(c[i].sum(0)) for i in idx]))
    rows = []
    for k in SYS:
        r = {'view': view, 'system': k, 'n_rec': int(m.sum()), 'n_sent': ncl, 'BLEU': point[k][0], 'BLEU_lo': ci(boot[k][0])[0], 'BLEU_hi': ci(boot[k][0])[1],
             'chrF++': point[k][1], 'chrF_lo': ci(boot[k][1])[0], 'chrF_hi': ci(boot[k][1])[1], 'ref': REFSYS[k]}
        if REFSYS[k]:
            for j, nm in enumerate(['dBLEU', 'dchrF++']):
                d = boot[k][j] - boot[REFSYS[k]][j]; r[nm] = point[k][j] - point[REFSYS[k]][j]; r[nm + '_lo'], r[nm + '_hi'] = ci(d)
        rows.append(r)
    return pd.DataFrame(rows)

t_v2 = pd.concat([eval_mt(v) for v in VIEWS], ignore_index=True)
TAB = os.path.join(OUT_DIR, 'tables'); os.makedirs(TAB, exist_ok=True); atomic_csv(t_v2, os.path.join(TAB, 'mt_scores_v2.csv'))
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.options.display.float_format = '{:.2f}'.format
cols = ['system', 'BLEU', 'BLEU_lo', 'BLEU_hi', 'chrF++', 'chrF_lo', 'chrF_hi', 'dBLEU', 'dBLEU_lo', 'dBLEU_hi', 'dchrF++', 'ref']
for v in VIEWS: print(f'\n== {v} =='); print(t_v2[t_v2['view'] == v][cols].to_string(index=False))

# lặp vòng và độ dài (bản ghi <= 30 s)
def has_rep(s):
    w = s.lower().split()
    return len(w) >= 8 and max(Counter(zip(w, w[1:], w[2:])).values()) >= 3
mk = (df.duration_sec <= MAX_SEC).values; rl = df.en_script.str.split().str.len().values[mk]
print('\n== Lặp vòng và độ dài (bản ghi <= 30 s) ==')
for k, v in SYS.items():
    v = np.array(v)[mk]
    print(f'{k:5s} | câu có cụm 3 từ lặp >= 3 lần: {sum(has_rep(s) for s in v):3d} | số từ dịch / số từ tham chiếu: {sum(len(s.split()) for s in v) / rl.sum():.3f}')


== pooled_main ==
system  BLEU  BLEU_lo  BLEU_hi  chrF++  chrF_lo  chrF_hi  dBLEU  dBLEU_lo  dBLEU_hi  dchrF++  ref
  V1_A 16.33    15.74    16.96   42.63    42.13    43.14    NaN       NaN       NaN      NaN None
  V2_A 16.42    15.81    17.03   42.67    42.18    43.17   0.10      0.01      0.20     0.04 V1_A
  V1_B 17.24    16.64    17.88   44.34    43.85    44.86   0.91      0.50      1.35     1.71 V1_A
  V2_B 17.38    16.77    17.98   44.40    43.91    44.91   0.14      0.04      0.25     0.05 V1_B

== pooled_all ==
system  BLEU  BLEU_lo  BLEU_hi  chrF++  chrF_lo  chrF_hi  dBLEU  dBLEU_lo  dBLEU_hi  dchrF++  ref
  V1_A 16.29    15.70    16.92   42.59    42.09    43.09    NaN       NaN       NaN      NaN None
  V2_A 16.38    15.80    17.02   42.63    42.14    43.14   0.09      0.00      0.19     0.04 V1_A
  V1_B 17.19    16.54    17.84   44.29    43.80    44.81   0.90      0.49      1.33     1.71 V1_A
  V2_B 17.33    16.70    17.97   44.35    43.85    44.86   0.14      0.04      0.

## 5. Ví dụ và lưu kết quả

In [8]:
chg = df[(df.hyp_B.values != V2['B'])]
print(f'V2_B khác V1_B ở {len(chg):,}/{len(df):,} câu; V2_A khác V1_A ở {int((df.hyp_A.values != V2["A"]).sum()):,} câu.')
rep_rows = [i for i, s in enumerate(df.hyp_B) if has_rep(s)][:3]
for i in rep_rows:
    print('\n- REF  :', df.en_script[i]); print('  ASR  :', df.asr_text[i]); print('  V1_B :', df.hyp_B[i]); print('  V2_B :', V2['B'][i])

out = df[['uid', 'split', 'fleurs_id', 'duration_sec', 'vi_script', 'en_script', 'asr_text']].copy()
for k, v in SYS.items(): out[f'hyp_{k}'] = v
atomic_csv(out, os.path.join(OUT_DIR, 'fleurs_v2_outputs.csv'))
try:
    from google.colab import drive
    drive.flush_and_unmount(); print('\nĐã flush Drive.')
except Exception as e:
    print('Bỏ qua flush:', e)

V2_B khác V1_B ở 373/4,212 câu; V2_A khác V1_A ở 354 câu.

- REF  : It is an archipelago with 15 islands spread out over 2.2 million km2 of ocean.
  ASR  : đó là một quần đảo bao gồm mười lăm đảo trải dài suốt một vùng biển rộng hai phẩy hai triệu cây số vuông.
  V1_B : It's an island that consists of fifteen islands spread across a two - point - two - and - a - square - square - square - square sea area.
  V2_B : It's an island of fifteen islands spread through a two - point - two - and - a - square - square square - meter sea area.

- REF  : There are many different film formats that have been used over the years. Standard 35 mm film (36 by 24 mm negative) is much the commonest.
  ASR  : có nhiều loại phim khác nhau đã được sử dụng trong những năm qua trong đó phim ba mươi lăm mi li mét tiêu chuẩn phim âm bản cỡ ba mươi sáu nhân hai tư mi li mét là loại thông dụng nhất.
  V1_B : There have been many different types of movies that have been used over the past few years in which the th